# Week 11 Live Coding: Can You Trust the Analyst?

The case told you to ask the analyst for her track record. She sent it. Today we 1) **check whether her probabilities mean what they say**, using a calibration table and a Brier score, and 2) **build the decision** the board has to make, and find what it actually turns on.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rec = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk11_forecasts_and_calibration/data/analyst_record.csv')
rec.head()

Each row is one race she forecast before this one. `forecast` is the probability she gave the candidate; `won` is 1 if that candidate won and 0 if not. 150 races, going back to 2018.

## Part 1: Do her numbers mean what they say?

Remember the test. Of all the times she said 30\%, did it happen about 30\% of the time? Start with one band, by hand.

In [ ]:
# Of the races she called around 80%, how many actually happened?
band = (rec['forecast'] >= 0.75) & (rec['forecast'] <= 0.85)
print('races in this band:', band.sum())
print('she said, on average:', round(rec.loc[band, 'forecast'].mean(), 3))
print('actually happened:   ', round(rec.loc[band, 'won'].mean(), 3))

She said about 0.79. It happened about 0.60. One band is not a verdict, so do all of them.

Two tools, one line each:
- **`pd.cut(...)`** sorts each forecast into a *bin*. `np.arange(0, 1.01, 0.2)` makes the edges 0, 0.2, 0.4, 0.6, 0.8, 1.0, which is **5 bins**.
- **`groupby('bin').agg(...)`** then averages *within* each bin: what she said, and what fraction actually happened.

If she is calibrated, those two columns match, bin by bin.

In [ ]:
rec['bin'] = pd.cut(rec['forecast'], bins=np.arange(0, 1.01, 0.2))

# YOUR CODE HERE (we will type this together): group by 'bin' and compute the mean
# forecast, the actual fraction that happened, and how many races are in each bin.
calibration = (rec.groupby('bin', observed=True)
               .agg(she_said=('forecast', 'mean'),
                    actually_happened=('won', 'mean'),
                    n=('won', 'size')))

print(calibration.round(3))

*You should see 27 / 36 / 26 / 41 / 20 races in the five bins.*

Read down the two columns. The first three bins land almost exactly on what she said. The last two do not: she says 0.72 and it happens 0.61; she says 0.88 and it happens 0.75.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], '--', color='gray', label='perfect calibration')
se = np.sqrt(calibration['actually_happened'] * (1 - calibration['actually_happened'])
             / calibration['n'])
ax.errorbar(calibration['she_said'], calibration['actually_happened'], yerr=1.96 * se,
            fmt='none', ecolor='#0F4D92', alpha=0.5, zorder=2)
ax.scatter(calibration['she_said'], calibration['actually_happened'],
           s=calibration['n'] * 4, color='#0F4D92', zorder=3,
           label='her record (dot size = races)')
ax.set_xlabel('What she forecast')
ax.set_ylabel('What actually happened')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

Same axes as Silver's chart, error bars and all. Her top two dots sit **below** the line, which is what overconfident looks like.

Now look at the bars. **Not one bin on its own is far enough from the line to be sure about** — Silver warns about exactly this on p. 2 of the reading: ten events at 80\% could come in anywhere by luck alone. That is why we pool them below.

### Does she beat just guessing?

Calibration is not everything. A forecaster who says the base rate every time is perfectly calibrated and useless. The Brier score checks whether she is actually telling you anything.

In [ ]:
brier_analyst = np.mean((rec['forecast'] - rec['won'])**2)
base_rate = rec['won'].mean()
brier_baseline = np.mean((base_rate - rec['won'])**2)

print('base rate (how often anyone wins):', round(base_rate, 3))
print('Brier, her forecasts:            ', round(brier_analyst, 4))
print('Brier, always guess the base rate:', round(brier_baseline, 4))

**0.212 against 0.248.** She beats the base rate, so she is not just guessing. But only just: that gap is right at the edge of what 150 races can establish (a paired test gives $p = 0.05$). Do not oversell it.

Now the part that matters for the board. Split her record at 60\%, because that is where the case's numbers live.

In [ ]:
low  = rec[rec['forecast'] <= 0.60]
high = rec[rec['forecast'] >  0.60]   # > not >=, so all 150 races are covered

for name, part in [('60% and below', low), ('above 60%   ', high)]:
    print(f'{name}: {len(part)} races, she said {part.forecast.mean():.3f}, '
          f'happened {part.won.mean():.3f}')

**This is the finding to take to the board.** Above 60\% she is off by about 12 points. At 60\% and below she is off by less than half a point.

Her forecast for our race is 60/40. Both numbers sit in the range where her record is good. You asked for the track record, you got it, and it did **not** give you a reason to throw out the 60\%.

## Part 2: The decision

Now build the board's decision in code. Four inputs, from the case.

In [ ]:
p_A, gen_A, signs_A = 0.60, 0.45, 0.40   # A: primary, November, share she signs
p_B, gen_B, signs_B = 0.40, 0.55, 1.00   # B: same three

ev_A = p_A * gen_A * signs_A
ev_B = p_B * gen_B * signs_B
print('A delivers, in expectation:', round(ev_A * 100, 2), '% of your ordinance')
print('B delivers, in expectation:', round(ev_B * 100, 2), '%')
print('stay neutral and you get:  ', round((ev_A + ev_B) * 100, 2), '%')

10.8, 22.0, and 32.8 — the three numbers from the slides.

Endorsing does not pick the mayor. It moves probability from A to B, and it annoys A. Write that as a function of `d`, the points of primary chance you move.

In [ ]:
def endorse_B(d, signs_A_annoyed=0.30):
    """What you end up with if you endorse B and move d points of primary chance."""
    a = (p_A - d) * gen_A * signs_A_annoyed
    b = (p_B + d) * gen_B * signs_B
    return a + b

for d in [0.00, 0.06, 0.10]:
    print(f'move {d*100:4.0f} points -> {endorse_B(d)*100:.2f}%')
print(f'stay neutral   -> {(ev_A + ev_B)*100:.2f}%')

30.10, 32.59, 34.25, against 32.80 for doing nothing. Six points is still not enough. Somewhere between 6 and 10 you draw level, so find it.

In [ ]:
neutral = ev_A + ev_B

# YOUR CODE HERE (together): you start (neutral - endorse_B(0)) behind, and each point
# you move buys endorse_B(0.01) - endorse_B(0). Divide one by the other.
behind    = neutral - endorse_B(0)
per_point = endorse_B(0.01) - endorse_B(0)

print('you start behind by:', round(behind * 100, 2), 'points')
print('each point buys:    ', round(per_point * 100, 3), 'points')
print('break-even:         ', round(behind / per_point, 1), 'points of primary chance')

*Check: 2.7 behind, 0.415 a point, break-even 6.5.*

### What is this answer actually resting on?

We invented the 30\%. Sweep it and watch the break-even move.

In [ ]:
print('if A signs this much   break-even')
for s in [0.40, 0.35, 0.30, 0.25, 0.20]:
    behind_s = neutral - endorse_B(0, signs_A_annoyed=s)
    per_pt_s = endorse_B(0.01, signs_A_annoyed=s) - endorse_B(0, signs_A_annoyed=s)
    print(f'        {s:.2f}              {behind_s / per_pt_s:5.1f} points')

At 0.40 — A does not mind at all — the break-even is **0.0**. Every point you move pays and there is nothing to decide.

So the whole question exists because of a number we made up. That is what the memo should be about.